# 법률저널 로스쿨 섹션 뉴스 스크래핑

- **Part 1**: 1~5페이지 기사 100개 수집 → `lawschool_news1.xlsx`
- **Part 2**: 처음 10개 기사 본문 수집 → `lawschool_news2.xlsx`

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

## Part 1: 기사 목록 수집 (1~5페이지, 100개)

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

BASE_URL = 'https://www.lec.co.kr'
LIST_URL = BASE_URL + '/news/articleList.html'

# 로스쿨 섹션 코드
SECTION_CODE = 'S1N4'

In [ ]:
titles = []
descriptions = []
links = []
writers = []
times = []

for page in range(1, 6):  # 1~5페이지
    params = {
        'sc_section_code': SECTION_CODE,
        'view_type': 'sm',
        'page': page
    }
    res = requests.get(LIST_URL, params=params, headers=headers, timeout=10)
    res.encoding = 'utf-8'
    soup = BeautifulSoup(res.text, 'html.parser')

    # 기사 목록 아이템: ul.type2 > li
    articles = soup.select('ul.type2 > li')
    print(f'Page {page}: {len(articles)} articles found')

    for article in articles:
        # 제목 및 링크
        title_el = article.select_one('h4.titles > a')
        if title_el is None:
            title_el = article.select_one('.titles a')
        title = title_el.get_text(strip=True) if title_el else ''
        href = title_el['href'] if title_el else ''
        link = BASE_URL + href if href and not href.startswith('http') else href

        # 도입부 (description)
        desc_el = article.select_one('.lead')
        if desc_el is None:
            desc_el = article.select_one('p.summary')
        description = desc_el.get_text(strip=True) if desc_el else ''

        # 바이라인: "섹션 | 기자이름 기자 | 게재시간"
        # 힌트: "|"를 구분자로 자를 때는 escape 필요 (\\|)
        byline_el = article.select_one('.byline')
        byline = byline_el.get_text(strip=True) if byline_el else ''
        parts = re.split(r'\|', byline)  # "|"로 분리

        writer = ''
        time_str = ''
        if len(parts) >= 3:
            # writer: "홍길동 기자" → "기자"와 공백 제거
            writer = re.sub(r'기자|\s', '', parts[1])
            time_str = parts[2].strip()
        elif len(parts) == 2:
            writer = re.sub(r'기자|\s', '', parts[0])
            time_str = parts[1].strip()

        titles.append(title)
        descriptions.append(description)
        links.append(link)
        writers.append(writer)
        times.append(time_str)

    time.sleep(1)  # 요청 간 1초 대기

print(f'\n총 수집 기사 수: {len(titles)}')

In [ ]:
# 데이터프레임 생성
df1 = pd.DataFrame({
    'title': titles,
    'description': descriptions,
    'link': links,
    'writer': writers,
    'time': times
})

df1.head(10)

In [ ]:
# lawschool_news1.xlsx 저장
df1.to_excel('lawschool_news1.xlsx', index=False)
print('lawschool_news1.xlsx 저장 완료')
print(df1.shape)

## Part 2: 처음 10개 기사 본문(content) 수집

In [ ]:
contents = []

for i, link in enumerate(df1['link'].head(10)):
    print(f'[{i+1}/10] 수집 중: {link}')
    try:
        res = requests.get(link, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, 'html.parser')

        # <p> 태그에 포함된 내용 추출 후 하나의 string으로 합치기
        # R 힌트: str_c(x, collapse = " ")
        paragraphs = soup.find_all('p')
        content = ' '.join([p.get_text() for p in paragraphs])
        contents.append(content)
    except Exception as e:
        print(f'  오류: {e}')
        contents.append('')

    time.sleep(1)  # 요청 간 1초 대기

print('\n본문 수집 완료')

In [ ]:
# content 컬럼을 마지막에 추가
# 처음 10개에만 content, 나머지는 None
content_col = contents + [None] * (len(df1) - 10)
df2 = df1.copy()
df2['content'] = content_col

df2.head(10)

In [ ]:
# lawschool_news2.xlsx 저장
df2.to_excel('lawschool_news2.xlsx', index=False)
print('lawschool_news2.xlsx 저장 완료')
print(df2.shape)